# Virtual IAI office

This simulation showcases a Tortugabot navigating and scanning IAI office environment using LiDAR sensor and camera. An environment for teaching sensors and navigation modules.

**Environment**: [IAI office](https://github.com/code-iai/iai_office_sim)

**Robot**: [Tortugabot](https://github.com/code-iai/tortugabot)

**Simulator**: [Gazebo](https://classic.gazebosim.org/)

---

<button data-commandlinker-command="notebook:run-all-cells" class="jupyter-button" style="color: #fff;font-size:1rem; background-color: #1976d2;">Run All Code</button>
<button data-commandlinker-command="notebook:interrupt-kernel" class="jupyter-button" style="font-size:1rem;">Stop All</button>

## Open virtual display

In [ ]:
%run ../utils.py
display_desktop(anchor='split-right')
sleep(3)

## Launch Simulation

You will see the Rviz window popup visualize the sensor data in red dots and a camera view on the left bottom.

In [ ]:
from psutil import Popen
import subprocess

Popen(['/bin/bash', '-c',
              'ros2 launch iai_office_sim turtlebot3.launch.py gz_gui:=false'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL)

Popen(['/bin/bash', '-c', 
              'ros2 launch slam_toolbox online_async_launch.py use_sim_time:=True'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL)

## Robot steering

The following code will display two slider to control the robot.

You can move the robot around to explore the simulation environment.

In [ ]:
velocity_publisher = VelocityPublisher()
robot_steering(velocity_publisher)

# Wait for simulator ready
sleep(3)

## Random walk algorithm

In [ ]:
# An simple example of random walk
import rclpy
from rclpy.node import Node
from sensor_msgs.msg import LaserScan
import numpy as np

# Detect obstacles in a certain direction
def detect_obstacle(msg, direction=0, min_dis=0.8, angle=40):
    ranges = np.roll(np.array(msg.ranges), np.ceil(direction + angle/2).astype(int))
    distances = ranges[:angle]
    return np.min(distances) < min_dis

# Random walk algorithm
# This is a callback function of the laserscan, will be called every 0.2 seconds.
def radom_walk(msg):
    hit_obstacle = detect_obstacle(msg)
    linear_x = velocity_publisher.cmd_vel_msg.linear.x
    angular_z = velocity_publisher.cmd_vel_msg.angular.z
    if hit_obstacle:
        linear_x = 0
        # set steering velocity randomly between -0.5 to 0.5
        if angular_z == 0:
            angular_z = np.random.rand() - 0.5
    else:
        linear_x = 0.3
        angular_z = 0
    velocity_publisher.publish_velocity(linear_x=linear_x, angular_z=angular_z)
    
print("Press the stop button to interrupt.")
# start a message subscriber
laserscan_subscriber = LaserScanSubscriber(callback=radom_walk)
try:
    rclpy.spin(laserscan_subscriber)
except KeyboardInterrupt:
    rclpy.logging.get_logger("Quitting `rclpy.spin(laserscan_subscriber)`").info('Laser Data Listening Stopped!')
# Destroy the subscriber node
laserscan_subscriber.subscription.destroy()
laserscan_subscriber.destroy_node()
# Stop robot
velocity_publisher.publish_velocity(linear_x=0, angular_z=0)
print("Robot stopped!")

**Save the map**

```bash
ros2 service call /slam_toolbox/save_map slam_toolbox/srv/SaveMap "name:
    data: 'my_map'"
```

You will find two files `my_map.pgm` and `my_map.yaml`.